# Superponer una trayectoria sobre el mapa energético

Este ejemplo conecta una simulación de Gillespie con el mapa bidimensional. La asociación estado \(ightarrow q\) se define explícitamente en el notebook: es una coordenada de visualización elegida por el usuario, no una coordenada estructural extraída del modelo cinético.

In [ ]:
from pathlib import Path
import sys

# Find the project root whether JupyterLab was launched from the repository
# root or from the notebooks/ folder.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / "src" / "protein_motor_sim").is_dir()),
    None,
)
if PROJECT_ROOT is not None:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT or "package expected to be installed in this kernel")

## 1. Simular kinesina y definir las coordenadas conformacionales ilustrativas

In [ ]:
import matplotlib.pyplot as plt
from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.mechanics.energy_landscape import EnergyLandscape

trajectory = GillespieSimulator().run(
    Kinesin1(),
    Environment(atp_concentration=1.0e-3, external_force_N=0.8e-12, temperature_K=310.0),
    t_max=0.25,
    seed=31,
)

# q is illustrative: it is not measured by the present kinetic model.
state_coordinates = {
    "bound_apo": 0.0,
    "atp_bound": 0.35,
    "post_hydrolysis": 1.0,
    "detached": 0.5,
}
landscape = EnergyLandscape(opposing_force_pN=0.8, temperature_K=310.0)
print("States in this realization:", sorted(set(trajectory.states)))

## 2. Mapa de energía y trayectoria

In [ ]:
from protein_motor_sim.visualization.energy_maps import plot_trajectory_on_energy_landscape

ax, scatter = plot_trajectory_on_energy_landscape(
    trajectory,
    landscape,
    state_coordinates=state_coordinates,
    x_range_nm=(min(trajectory.positions_nm)-12, max(trajectory.positions_nm)+12),
)
plt.show()

## 3. Tabla de correspondencia estado-coordenada

In [ ]:
import pandas as pd

pd.DataFrame(
    [{"kinetic_state": state, "illustrative_q": q} for state, q in state_coordinates.items()]
)